# 0.2 Xatu Calldata Pull

This notebook is the canonical calldata source for the bandwidth pipeline. It pulls total raw calldata bytes from Xatu's canonical beacon execution-payload transaction table, then validates zero/nonzero byte counts from Xatu's execution transaction tables.

Output:

```text
calldata_bytes = sum(canonical_beacon_block_execution_transaction.call_data_size)
calldata_gas_7999 = sum(4 * n_input_zero_bytes + 16 * n_input_nonzero_bytes)
```

## Why Xatu for Calldata

Xatu has full payload transaction coverage for calldata, and it is much cheaper to query at scale than RPC. Raw bytes come from the beacon payload transaction table. Zero/nonzero byte counts come from `execution_transaction` only after validating that it matches the beacon payload transaction count and raw byte total. `canonical_execution_transaction` is diagnostic only. RPC remains the source for BAL bytes because exact BAL reads need `prestateTracer`.

In [1]:
import os
from pathlib import Path

import clickhouse_connect
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sim.xatu_calldata import query_xatu_calldata_by_block

load_dotenv(PROJECT_ROOT / ".env")
missing = [name for name in ["CLICKHOUSE_USER", "CLICKHOUSE_PASSWORD"] if not os.environ.get(name)]
if missing:
    raise RuntimeError("Missing .env values: " + ", ".join(missing))

client = clickhouse_connect.get_client(
    host="clickhouse-raw.xatu.ethpandaops.io",
    port=443,
    secure=True,
    username=os.environ["CLICKHOUSE_USER"],
    password=os.environ["CLICKHOUSE_PASSWORD"],
)
print(client.query("SELECT version()").result_rows)

[('26.2.5.45',)]


In [2]:
NETWORK = "mainnet"
START_BLOCK = 22_886_891
N_BLOCKS = 50
BLOCKS = list(range(START_BLOCK, START_BLOCK + N_BLOCKS))
WRITE_CSV = True

In [3]:
calldata = query_xatu_calldata_by_block(client, BLOCKS, network=NETWORK)

calldata["execution_tx_row_delta"] = calldata["execution_tx_rows"] - calldata["n_txs_from_payload"]
calldata["execution_calldata_delta"] = calldata["execution_calldata_bytes"] - calldata["calldata_bytes"]

display(calldata)

summary = pd.DataFrame([{
    "blocks_checked": len(calldata),
    "execution_matches": int(calldata["execution_matches_beacon"].sum()),
    "canonical_execution_matches": int(calldata["canonical_execution_matches_beacon"].sum()),
    "total_payload_txs": int(calldata["n_txs_from_payload"].sum()),
    "total_calldata_bytes": int(calldata["calldata_bytes"].sum()),
    "total_zero_bytes": int(calldata["calldata_zero_bytes"].dropna().sum()),
    "total_nonzero_bytes": int(calldata["calldata_nonzero_bytes"].dropna().sum()),
    "total_calldata_gas_7999": int(calldata["calldata_gas_7999"].dropna().sum()),
    "mean_calldata_bytes_per_block": calldata["calldata_bytes"].mean(),
    "mean_calldata_gas_per_block": calldata["calldata_gas_7999"].dropna().mean(),
}])
display(summary)

if WRITE_CSV:
    data_dir = PROJECT_ROOT / "data"
    data_dir.mkdir(exist_ok=True)
    out = data_dir / f"xatu_calldata_{min(BLOCKS)}_{max(BLOCKS)}.csv"
    calldata.to_csv(out, index=False)
    print(out)

,block_number,slot,n_txs_from_payload,n_txs,calldata_bytes,execution_tx_rows,execution_calldata_bytes,calldata_zero_bytes,calldata_nonzero_bytes,calldata_gas_7999,calldata_gas_source,execution_rows_n_input_positive,execution_matches_beacon,canonical_execution_tx_rows,canonical_execution_matches_beacon,execution_tx_row_delta,execution_calldata_delta
0,22886891,12108642,268,268,63197,268,63197,39716,23481,534560,execution_transaction,185,True,184,False,0,0
1,22886892,12108643,183,183,76877,183,76877,54798,22079,572456,execution_transaction,133,True,129,False,0,0
2,22886893,12108644,162,162,78355,162,78355,55493,22862,587764,execution_transaction,121,True,118,False,0,0
3,22886894,12108645,122,122,40638,122,40638,27441,13197,320916,execution_transaction,79,True,80,False,0,0
4,22886895,12108646,223,223,56590,223,56590,39140,17450,435760,execution_transaction,145,True,147,False,0,0
5,22886896,12108647,282,282,65928,282,65928,46763,19165,493692,execution_transaction,205,True,174,False,0,0
6,22886897,12108648,155,155,35659,155,35659,21937,13722,307300,execution_transaction,117,True,96,False,0,0
7,22886898,12108649,238,238,78196,238,78196,51743,26453,630220,execution_transaction,143,True,157,False,0,0
8,22886899,12108650,165,165,43392,165,43392,26726,16666,373560,execution_transaction,88,True,110,False,0,0
9,22886900,12108651,263,263,54982,263,54982,37451,17531,430300,execution_transaction,131,True,187,False,0,0


,blocks_checked,execution_matches,canonical_execution_matches,total_payload_txs,total_calldata_bytes,total_zero_bytes,total_nonzero_bytes,total_calldata_gas_7999,mean_calldata_bytes_per_block,mean_calldata_gas_per_block
0,50,50,0,9658,2850752,1896807,953945,22850348,57015.04,457006.96


/Users/william/PycharmProjects/eip-7999-research/data/xatu_calldata_22886891_22886940.csv
